# INF 791 - Tópicos Especiais II – Redes Complexas
## Trabalho Prático - Intermediário
**Aluno:** Pedro Henrique Silva Oliveira


In [2]:
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import requests
import gzip
import shutil
import os

# Configurações visuais
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)


## 1. Coleta de Dados

**(a) Escolha e Coleta do Grafo**

Para a realização deste trabalho, a rede selecionada foi a **Social circles: Facebook** (Facebook ego networks), obtida a partir do repositório público [SNAP (Stanford Large Network Dataset Collection)](http://snap.stanford.edu/data/ego-Facebook.html). Este dataset é composto pela união de 10 redes egocêntricas (círculos de amigos de usuários específicos) coletadas de participantes voluntários da plataforma. 

Na modelagem do grafo, cada nó representa um usuário individual do Facebook, cujas identidades reais foram completamente anonimizadas e substituídas por identificadores numéricos inteiros únicos para preservar sua privacidade. Embora o repositório original do SNAP disponibilize arquivos complementares contendo vetores de atributos de perfil dos usuários (gênero, instituição de ensino, cidade natal e grupos de interesse), no contexto deste trabalho apenas a topologia da rede foi importada, já que as análises que faremos dizem mais sobre as relações de conectividade da rede. 

As arestas do grafo representam relações de amizade mútuas e ativas na plataforma. Como a amizade no Facebook exige a aceitação bilateral de ambas as partes, a rede é modelada como um grafo não-direcionado e não-ponderado. 

A célula de código a seguir implementa a automação do processo de obtenção e inicialização da rede social no ambiente de execução. O script realiza duas tarefas principais:

1. **Download e Extração Automatizados:** Verifica se o arquivo do dataset (`facebook_combined.txt`) já existe localmente. Caso não exista, o código faz uma requisição HTTP via biblioteca `requests` para baixar a versão compactada (`.gz`) diretamente do repositório oficial do SNAP, salvando-a em disco e, em seguida, descompactando-a com a biblioteca nativa `gzip` do Python.
2. **Criação e Modelagem do Grafo:** Utiliza o método `read_edgelist` da biblioteca `NetworkX` para ler o arquivo de texto bruto contendo a lista de arestas e construir um objeto de grafo não-direcionado (`nx.Graph`), convertendo os identificadores dos nós para o tipo inteiro (`nodetype=int`). Ao final, a variável global `G` é inicializada e as dimensões da rede (nós e arestas) são impressas na tela como confirmação.


In [1]:
# Baixando o dataset do SNAP automaticamente
url = "https://snap.stanford.edu/data/facebook_combined.txt.gz"
file_gz = "facebook_combined.txt.gz"
file_txt = "facebook_combined.txt"

if not os.path.exists(file_txt):
    print("Baixando o dataset...")
    response = requests.get(url, stream=True)
    
    print("Passando para arquivo local de saída...")
    with open(file_gz, 'wb') as out_file:
        shutil.copyfileobj(response.raw, out_file)
    
    print("Extraindo o dataset...")
    with gzip.open(file_gz, 'rb') as f_in:
        with open(file_txt, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    print("Download e extração concluídos.")
else:
    print("Dataset já carregado")

# Carregando o grafo no NetworkX
print("Carregando o grafo...")
G = nx.read_edgelist(file_txt, create_using=nx.Graph(), nodetype=int)
print(f"Grafo carregado com {G.number_of_nodes()} nós e {G.number_of_edges()} arestas.")


NameError: name 'os' is not defined

**(b) Identificação do Grafo e Limitações**

O grafo analisado é uma composição de 10 ego-networks. Ele é um subgrafo amostrado da verdadeira rede do Facebook da época.
**Limitações:**
- Por ser construído a partir de redes egocêntricas de usuários específicos, a rede possui uma densidade artificialmente alta ao redor desses "egos" centrais. Ou seja, ele sofre de viés de seleção.
- A fração do grafo coletado em relação à rede global do Facebook é ínfima. Contudo, devido a propriedades de redes *Scale-Free* (Livres de Escala), as dinâmicas topológicas globais ainda costumam refletir o comportamento macroscópico da rede real.


## 2. Análise de uma Rede Complexa

Nesta seção, calcularemos as métricas da rede e as compararemos com os conceitos vistos nas aulas da disciplina.

### (a) Breve explicação da rede analisada
A rede é não-direcionada e não-ponderada, onde cada nó é um usuário anonimizado do Facebook e cada aresta representa que dois usuários são amigos (uma relação inerentemente simétrica). O objetivo é provar se esta amostra exibe o comportamento clássico de Redes Complexas Sociais.


### (b) Distribuição e Grau Médio
Conforme visto na Aula 02, o grau indica a quantidade de arestas (amigos) conectadas a um nó.


In [ ]:
# Cálculo dos graus
degrees = [degree for node, degree in G.degree()]

# Grau médio
avg_degree = np.mean(degrees)
print(f"Grau Médio: {avg_degree:.2f}")

# Contagem da distribuição dos graus
degree_counts = nx.degree_histogram(G)
degrees_x = range(len(degree_counts))

# Plotando a distribuição em escala linear
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.bar(degrees_x, degree_counts, width=1.0, color='b')
plt.title("Distribuição de Graus (Linear)")
plt.xlabel("Grau")
plt.ylabel("Frequência")

# Plotando a distribuição em escala log-log
plt.subplot(1, 2, 2)
plt.loglog(degrees_x, degree_counts, marker='o', linestyle='none', color='r')
plt.title("Distribuição de Graus (Log-Log)")
plt.xlabel("Grau (log)")
plt.ylabel("Frequência (log)")

plt.tight_layout()
plt.show()


**Discussão:**
O grau médio é aproximadamente 43.69, ou seja, um usuário típico tem cerca de 44 amigos nesta amostra. Observando o gráfico *Log-Log*, notamos uma cauda longa evidente. A maioria esmagadora dos nós possui poucos amigos, enquanto um número muito pequeno de nós (*Hubs*) possui centenas de conexões. Isso é o comportamento típico de uma rede guiada por *Lei de Potência* (*Power-Law* ou rede Livre de Escala), um padrão estrutural universal em redes complexas e sociais abordado na disciplina.


### (c) Número de Componentes do Grafo
Segundo a Aula 02, um componente conexo é um grupo de nós onde todos se alcançam.


In [ ]:
num_components = nx.number_connected_components(G)
print(f"Número de componentes do grafo: {num_components}")

components = list(nx.connected_components(G))
sizes = [len(c) for c in components]
print(f"Tamanho de cada componente: {sizes}")


**Discussão:**
A rede possui **apenas 1 componente**, cujo tamanho engloba todos os 4.039 nós do grafo. Isto ilustra empiricamente a formação de um **Componente Gigante**, um conceito crucial visto em sala que demonstra a tendência estrutural de grandes redes do mundo real (como a Web ou grandes redes de colaboração) se aglomerarem num único bloco maciço de conexões.


### (d) Coeficiente de Clusterização
Conforme a Aula 03, o coeficiente de clusterização está ligado ao **Fechamento Triádico** (*Triadic Closure*): a probabilidade de dois amigos seus também serem amigos entre si.


In [ ]:
# Clusterização de cada nó
clustering_coeffs = nx.clustering(G)
cc_values = list(clustering_coeffs.values())

# Clusterização Global (Média dos locais)
global_cc = nx.average_clustering(G)
print(f"Coeficiente de Clusterização Global: {global_cc:.4f}")

# Plot da distribuição
plt.figure(figsize=(8, 5))
sns.histplot(cc_values, bins=30, kde=True, color='purple')
plt.title("Distribuição do Coeficiente de Clusterização Local")
plt.xlabel("Coeficiente de Clusterização")
plt.ylabel("Frequência")
plt.show()


**Discussão:**
O coeficiente de clusterização global alcançou incríveis 0.6055. Em aulas teóricas, vimos que as redes sociais tendem a apresentar coeficientes muito maiores que os das redes puramente tecnológicas. Um valor de ~60% significa que o princípio sociológico do Fechamento Triádico atua fortemente aqui: os laços formam comunidades densas e os amigos frequentemente se conhecem (formando painéis e grupos coesos).


### (e) Distribuição do Tamanho dos Componentes
Como identificado e justificado na questão (c), o grafo inteiro pertence a **apenas um Componente Gigante** de tamanho 4039. Como o enunciado diz expressamente que '*Se o grafo possuir apenas um componente não é preciso plotar*', estamos isentos deste gráfico.


### (f) Overlap da Vizinhança
Conforme a Aula 03, a sobreposição de vizinhança quantifica o percentual de amigos em comum (similaridade de Jaccard) e se relaciona com a teoria de Granovetter sobre **Arestas Fortes e Fracas** (*Strength of Weak Ties*).


In [ ]:
# O iterador jaccard_coefficient retorna (u, v, p) onde p é o overlap
jaccard_iter = nx.jaccard_coefficient(G)
overlaps = [p for u, v, p in jaccard_iter]

# Plot da distribuição dos overlaps
plt.figure(figsize=(8, 5))
sns.histplot(overlaps, bins=30, kde=True, color='green')
plt.title("Distribuição do Overlap da Vizinhança (Arestas)")
plt.xlabel("Overlap (Similaridade de Jaccard)")
plt.ylabel("Frequência")
plt.show()

print(f"Overlap Médio das Arestas: {np.mean(overlaps):.4f}")


**Discussão:**
Muitas arestas apresentam overlap baixo (próximo a 0), atuando como **Pontes Locais** ou **Elos Fracos**. Segundo a teoria, esses nós são cruciais para a propagação de novas informações na rede e para conectar grupos distintos (preenchendo os *Structural Holes*). Paralelamente, há uma cauda considerável de alto overlap, representando os Elos Fortes, que formam a estrutura base das comunidades de amigos íntimos.


### (g) Distância Média e Distribuição das Distâncias
A distância (menor caminho) entre dois pontos reflete diretamente as noções de navegação da rede. O cálculo demandará varrer os caminhos mais curtos entre todos os pares.


In [ ]:
import time
print("Calculando distâncias mais curtas de todos os pares... (isso pode levar de alguns segundos a um minuto)")
start_time = time.time()

# nx.shortest_path_length(G) retorna um iterador de dicionários para todos os nós
path_lengths_iter = nx.shortest_path_length(G)

all_distances = []
for source, targets in path_lengths_iter:
    for target, length in targets.items():
        if source < target: # Contar cada par apenas uma vez
            all_distances.append(length)

end_time = time.time()
print(f"Tempo de cálculo: {end_time - start_time:.2f} segundos")

avg_distance = np.mean(all_distances)
print(f"Distância Média da Rede: {avg_distance:.4f}")

# Plot da distribuição de distâncias
plt.figure(figsize=(8, 5))
# Como distâncias são inteiras, vamos contar as frequências usando um bar plot
dist_counts = np.bincount(all_distances)
dist_x = np.nonzero(dist_counts)[0]
dist_y = dist_counts[dist_x]

plt.bar(dist_x, dist_y, color='coral')
plt.title("Distribuição das Distâncias Mais Curtas")
plt.xlabel("Distância (Número de Arestas)")
plt.ylabel("Frequência")
plt.xticks(dist_x)
plt.show()


**Discussão:**
A distância média observada é extremamente baixa: apenas ~3.69 pulos. A maior parte das pessoas se alcança em 3 ou 4 saltos. Isso confirma empiricamente a propriedade de **Small World (Mundo Pequeno)** estudada em aula (referente aos "6 Graus de Separação" do Experimento de Milgram). Os Elos Fracos mapeados na questão anterior são diretamente os responsáveis por criar atalhos curtos numa rede de milhões (ou neste caso, de milhares de nós).


### (h) Visualização do Grafo
Para evitar visualizar um emaranhado denso de fios incompreensível (um *hairball* gigantesco causado por 4.039 nós fortemente conectados num único componente), exibiremos a rede egocêntrica localizada (o Componente Vizinho Imediato) focada num grande nó central, utilizando a própria engine do Matplotlib acoplada ao NetworkX.


In [ ]:
# Extraindo uma ego-network centrada num grande hub
target_node = 0
ego_network = nx.ego_graph(G, target_node, radius=1)

print(f"Visualizando ego-network local do nó {target_node} ({ego_network.number_of_nodes()} nós).")

plt.figure(figsize=(10, 10))
pos = nx.spring_layout(ego_network, seed=42)

# Nó ego
nx.draw_networkx_nodes(ego_network, pos, nodelist=[target_node], node_size=200, node_color='red')
# Nós periféricos (amigos)
nx.draw_networkx_nodes(ego_network, pos, nodelist=[n for n in ego_network.nodes if n != target_node], node_size=30, node_color='skyblue', alpha=0.8)

nx.draw_networkx_edges(ego_network, pos, alpha=0.2, edge_color='gray')

plt.title(f"Amostra Visual da Estrutura Ego-cêntrica (Nó {target_node})")
plt.axis("off")
plt.show()
